# exp022_distance_uncertainty_shrink train

Fold-safe OOF audit for fixed uncertainty-aware residual shrink candidates on top of the exp021 distance-weighted LightGBM profile.


## Contents

1. Setup and configuration
2. Selected weighted profile, uncertainty candidates, and inputs
3. Weighted OOF postprocess audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from distance_uncertainty_shrink import (
    experiment_config,
    run_weighted_oof_cv,
    selected_training_variant,
    train_files,
    write_metrics_from_summary,
)
from settings import EXPERIMENT_NAME, ExperimentPaths

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = experiment_config()

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)
print("Parent best variant:", config.get("audit", {}).get("parent_best_variant"))
print("Parent best CV:", config.get("audit", {}).get("parent_best_cv"))
print("Selected postprocess:", config.get("postprocess", {}).get("selected_method"))


## 2. Selected Weighted Profile, Uncertainty Candidates, And Inputs


In [ ]:
variant = selected_training_variant(config)
files = train_files(paths, max_wells=MAX_WELLS if DEBUG else MAX_WELLS)
print("Selected training variant:", variant["name"])
print("Weight profile:", variant.get("weight_profile"))
print("Train wells:", len(files))
print("Feature set:", config["model"]["feature_set"])
print("Estimator:", config["model"]["drift_model"]["estimator"])
print("Candidate postprocess methods:", config["postprocess"].get("candidate_methods"))
print("Distance buckets:", [bucket["name"] for bucket in config["audit"]["distance_buckets"]])


## 3. Weighted OOF Postprocess Audit

Each fold fits the selected exp020/exp021 weight profile only on training-fold wells. Postprocess candidates use only inference-time proxy features: row distance, tail progress, GR missingness, Z displacement, raw residual magnitude, and last-known anchor.


In [ ]:
summary = run_weighted_oof_cv(
    files,
    config,
    paths.artifacts_dir,
    max_wells=MAX_WELLS if DEBUG else MAX_WELLS,
)
write_metrics_from_summary(paths, summary)
print(json.dumps(summary, indent=2, sort_keys=True))


## 4. Metrics And Artifacts


In [ ]:
for path in sorted(paths.artifacts_dir.glob("*")):
    if path.is_file():
        print(path.name, path.stat().st_size)
print("metrics.json:", paths.metrics_path.read_text())
